# Problem 7, Bianchi identities (affine connection)

**Goal.** For an affine connection $\nabla$, syntactically close the
two Bianchi identities with the engine:

$$
\begin{aligned}
\text{Bianchi I:}\quad &
  \sum_{(U,V,W) \text{ cyc}} R(U, V) W
  \;=\;
  \sum_{(U,V,W) \text{ cyc}} \bigl[(\nabla_U T)(V, W) + T(T(U,V), W)\bigr], \\
\text{Bianchi II:}\quad &
  \sum_{(U,V,W) \text{ cyc}} (\nabla_U R)(V, W) Z
  \;=\;
  \sum_{(U,V,W) \text{ cyc}} R(U, T(V, W)) Z.
\end{aligned}
$$

Here $T(\nabla)(X, Y) = \nabla_X Y - \nabla_Y X - [X, Y]_{VF}$ is the
torsion and $R(\nabla)(X, Y) Z = \nabla_X \nabla_Y Z - \nabla_Y \nabla_X Z
- \nabla_{[X, Y]_{VF}} Z$ the curvature tensor.

**Approach.** `BianchiProblem` packages connection-linearity / Leibniz
rules + torsion/curvature definitions + tensor-Leibniz rules + LBVF
antisymmetry and Jacobi into a single `ExpansionEngine`.
`prove_first_bianchi(U, V, W)` builds LHS − RHS, expands with the engine,
and iterates `canonicalize` and `collect_terms` to a fixed point, if it
closes to zero, `ok=True`.

## Setup

We define a standard connection $\nabla$ and four vector fields and
spin up the wrapper.

In [16]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

In [17]:
from jacopy.algebra.derivation import Derivation
from jacopy.calculus.connection import connection
from jacopy.display.jupyter import display_chain
from jacopy.library import BianchiProblem
from jacopy.proof.chain import ProofChain

nabla = connection()
U = Derivation("U", 0)
V = Derivation("V", 0)
W = Derivation("W", 0)
Z = Derivation("Z", 0)

prob = BianchiProblem(nabla)
prob

BianchiProblem(∇)

## LHS / RHS shapes

Structural view of both sides, the right-hand side of Bianchi I has
six terms (three cyclic permutations × two packets:
$(\nabla_U T)(V,W)$ + $T(T(U,V), W)$), the left has three curvature
atoms.

In [18]:
lhs1 = prob.first_bianchi_lhs(U, V, W)
rhs1 = prob.first_bianchi_rhs(U, V, W)
print("Bianchi I LHS:", lhs1._repr_inner())
print("Bianchi I RHS:", rhs1._repr_inner())
print()
lhs2 = prob.second_bianchi_lhs(U, V, W, Z)
rhs2 = prob.second_bianchi_rhs(U, V, W, Z)
print("Bianchi II LHS:", lhs2._repr_inner())
print("Bianchi II RHS:", rhs2._repr_inner())

Bianchi I LHS: (R(∇)(U,V)(W) + R(∇)(V,W)(U) + R(∇)(W,U)(V))
Bianchi I RHS: ((∇_U T(∇))(V,W) + T(∇)(T(∇)(U,V),W) + (∇_V T(∇))(W,U) + T(∇)(T(∇)(V,W),U) + (∇_W T(∇))(U,V) + T(∇)(T(∇)(W,U),V))

Bianchi II LHS: ((∇_U R(∇))(V,W)(Z) + (∇_V R(∇))(W,U)(Z) + (∇_W R(∇))(U,V)(Z))
Bianchi II RHS: (R(∇)(U,T(∇)(V,W))(Z) + R(∇)(V,T(∇)(W,U))(Z) + R(∇)(W,T(∇)(U,V))(Z))


## Bianchi I, mechanical closure

`prove_first_bianchi` runs LHS − RHS through the engine; we check the
closure value and step count. `lhs_steps` records every axiom-fire the
engine performs as a `ProofStep`; `display_chain` opens the transcript
(canonicalize / collect_terms passes run silently in between, only the
engine steps appear in the transcript).

In [19]:
result1 = prob.prove_first_bianchi(U, V, W)
chain1 = ProofChain(result1.lhs_steps)
print("OK?", result1.ok)
print("final:", result1.lhs_final._repr_inner())
print(f"engine steps: {len(chain1)}")
display_chain(chain1)

OK? True
final: 0
engine steps: 60


{\allowdisplaybreaks\scriptsize
\begin{gather*}
R(∇)(U,V)(W) \to ∇_{U(∇_V(W))} - ∇_{V(∇_U(W))} - ∇_{[U,V]_{VF}(W)} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
R(∇)(V,W)(U) \to ∇_{V(∇_W(U))} - ∇_{W(∇_V(U))} - ∇_{[V,W]_{VF}(U)} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
R(∇)(W,U)(V) \to ∇_{W(∇_U(V))} - ∇_{U(∇_W(V))} - ∇_{[W,U]_{VF}(V)} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
[W,U]_{VF} \to -[U,W]_{VF} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
∇_{(-[U,W]_{VF})(V)} \to -∇_{[U,W]_{VF}(V)} \quad \text{[∇\_X X-linearity [∇]]\,(axiom)} \\
(∇_U T(∇))(V,W) \to ∇_U(T(∇)(V,W)) - T(∇)(∇_U(V),W) - T(∇)(V,∇_U(W)) \quad \text{[(∇\_U T)(V,W) = ∇\_U T(V,W) \ensuremath{-} T(∇\_U V,W) \ensuremath{-} T(V,∇\_U W) [∇]]\,(axiom)} \\
T(∇)(V,W) \to ∇_V(W) - ∇_W(V) - [V,W]_{VF} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
∇_{U((∇_{V(W) + (-∇_{W(V)) + (-[V,W]_{VF})))}}} \to ∇_{U(∇_V(W))} - ∇_{U(∇_W(V))} - ∇_{U([V,W]_{VF})} \quad \text{[∇\_X Y-additivity [∇]]\,(axiom)} \\
T(∇)(∇_U(V),W) \to ∇_{∇_U(V)(W)} - ∇_{W(∇_U(V))} - [∇_{U(V),W]_{VF}} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
[∇_{U(V),W]_{VF}} \to -[W,∇_{U(V)]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
T(∇)(V,∇_U(W)) \to ∇_{V(∇_U(W))} - ∇_{∇_U(W)(V)} - [V,∇_{U(W)]_{VF}} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
T(∇)(U,V) \to ∇_U(V) - ∇_V(U) - [U,V]_{VF} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
T(∇)((∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF})),W)}} \to ∇_{(∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF}))(W)}}} - ∇_{W((∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF})))}}} - [(∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF)),W]_{VF}}}} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
∇_{(∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF}))(W)}}} \to ∇_{∇_U(V)(W)} - ∇_{∇_V(U)(W)} - ∇_{[U,V]_{VF}(W)} \quad \text{[∇\_X X-linearity [∇]]\,(axiom)} \\
∇_{W((∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF})))}}} \to ∇_{W(∇_U(V))} - ∇_{W(∇_V(U))} - ∇_{W([U,V]_{VF})} \quad \text{[∇\_X Y-additivity [∇]]\,(axiom)} \\
[(∇_{U(V) + (-∇_{V(U)) + (-[U,V]_{VF)),W]_{VF}}}} \to [∇_{U(V),W]_{VF}} + [(-∇_{V(U)),W]_{VF}} + [(-[U,V]_{VF),W]_{VF}} \quad \text{[[Sum(a,b,\ensuremath{\dots}), Y]\_VF \ensuremath{\to} \ensuremath{\Sigma}\_i [a\_i, Y]\_VF  (and mirror)]\,(axiom)} \\
[∇_{U(V),W]_{VF}} \to -[W,∇_{U(V)]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
[(-∇_{V(U)),W]_{VF}} \to -[∇_{V(U),W]_{VF}} \quad \text{[[(\ensuremath{-}X), Y]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF]\,(axiom)} \\
[∇_{V(U),W]_{VF}} \to -[W,∇_{V(U)]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
[(-[U,V]_{VF),W]_{VF}} \to -[[U,V]_{VF,W]_{VF}} \quad \text{[[(\ensuremath{-}X), Y]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF]\,(axiom)} \\
[[U,V]_{VF,W]_{VF}} \to -[W,[U,V]_{VF]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
(∇_V T(∇))(W,U) \to ∇_V(T(∇)(W,U)) - T(∇)(∇_V(W),U) - T(∇)(W,∇_V(U)) \quad \text{[(∇\_U T)(V,W) = ∇\_U T(V,W) \ensuremath{-} T(∇\_U V,W) \ensuremath{-} T(V,∇\_U W) [∇]]\,(axiom)} \\
T(∇)(W,U) \to ∇_W(U) - ∇_U(W) - [W,U]_{VF} \quad \text{[T(∇)(X,Y) = ∇\_X Y \ensuremath{-} ∇\_Y X \ensuremath{-} [X,Y]\_VF [∇]]\,(axiom)} \\
[W,U]_{VF} \to -[U,W]_{VF} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
∇_{V((∇_{W(U) + (-∇_{U(W)) + (-(-[U,W]_{VF}))))}}} \to ∇_{V(∇_W(U))} - ∇_{V(∇_U(W))} - ∇_{V((-[U,W]_{VF}))

In [20]:
for i, step in enumerate(chain1.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(U,V)(W)
 -> (∇_U(∇_V(W)) + (-∇_V(∇_U(W))) + (-∇_[U,V]_VF(W)))

[2] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(V,W)(U)
 -> (∇_V(∇_W(U)) + (-∇_W(∇_V(U))) + (-∇_[V,W]_VF(U)))

[3] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(W,U)(V)
 -> (∇_W(∇_U(V)) + (-∇_U(∇_W(V))) + (-∇_[W,U]_VF(V)))

[4] [Y, X]_VF → −[X, Y]_VF  [repr-canonical] [axiom]
    [W,U]_VF
 -> (-[U,W]_VF)

[5] ∇_X X-linearity [∇] [axiom]
    ∇_(-[U,W]_VF)(V)
 -> (-∇_[U,W]_VF(V))

[6] (∇_U T)(V,W) = ∇_U T(V,W) − T(∇_U V,W) − T(V,∇_U W) [∇] [axiom]
    (∇_U T(∇))(V,W)
 -> (∇_U(T(∇)(V,W)) + (-T(∇)(∇_U(V),W)) + (-T(∇)(V,∇_U(W))))

[7] T(∇)(X,Y) = ∇_X Y − ∇_Y X − [X,Y]_VF [∇] [axiom]
    T(∇)(V,W)
 -> (∇_V(W) + (-∇_W(V)) + (-[V,W]_VF))

[8] ∇_X Y-additivity [∇] [axiom]
    ∇_U((∇_V(W) + (-∇_W(V)) + (-[V,W]_VF)))
 -> (∇_U(∇_V(W)) + (-∇_U(∇_W(V))) + (-∇_U([V,W]_VF)))

[9] T(∇)(X,Y) = ∇_X Y − ∇_Y X − [X,Y]_V

## Bianchi II, mechanical closure

Same protocol for the $(\nabla_U R)(V, W) Z$ cycle.

In [21]:
result2 = prob.prove_second_bianchi(U, V, W, Z)
chain2 = ProofChain(result2.lhs_steps)
print("OK?", result2.ok)
print("final:", result2.lhs_final._repr_inner())
print(f"engine steps: {len(chain2)}")
display_chain(chain2)

OK? True
final: 0
engine steps: 63


{\allowdisplaybreaks\scriptsize
\begin{gather*}
(∇_U R(∇))(V,W)(Z) \to ∇_U(R(∇)(V,W)(Z)) - R(∇)(∇_U(V),W)(Z) - R(∇)(V,∇_U(W))(Z) - R(∇)(V,W)(∇_U(Z)) \quad \text{[(∇\_U R)(V,W)Z = ∇\_U R(V,W)Z \ensuremath{-} R(∇\_U V,W)Z \ensuremath{-} R(V,∇\_U W)Z \ensuremath{-} R(V,W) ∇\_U Z [∇]]\,(axiom)} \\
R(∇)(V,W)(Z) \to ∇_{V(∇_W(Z))} - ∇_{W(∇_V(Z))} - ∇_{[V,W]_{VF}(Z)} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
∇_{U((∇_{V(∇_{W(Z)) + (-∇_{W(∇_{V(Z))) + (-∇_{[V,W]_{VF}(Z))))}}}}}} \to ∇_{U(∇_{V(∇_W(Z)))}} - ∇_{U(∇_{W(∇_V(Z)))}} - ∇_{U(∇_{[V,W]_{VF}(Z))}} \quad \text{[∇\_X Y-additivity [∇]]\,(axiom)} \\
R(∇)(∇_U(V),W)(Z) \to ∇_{∇_{U(V)(∇_W(Z))}} - ∇_{W(∇_{∇_U(V)(Z))}} - ∇_{[∇_{U(V),W]_{VF}(Z)}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
[∇_{U(V),W]_{VF}} \to -[W,∇_{U(V)]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
∇_{(-[W,∇_{U(V)]_{VF})(Z)}} \to -∇_{[W,∇_{U(V)]_{VF}(Z)}} \quad \text{[∇\_X X-linearity [∇]]\,(axiom)} \\
R(∇)(V,∇_U(W))(Z) \to ∇_{V(∇_{∇_U(W)(Z))}} - ∇_{∇_{U(W)(∇_V(Z))}} - ∇_{[V,∇_{U(W)]_{VF}(Z)}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
R(∇)(V,W)(∇_U(Z)) \to ∇_{V(∇_{W(∇_U(Z)))}} - ∇_{W(∇_{V(∇_U(Z)))}} - ∇_{[V,W]_{VF(∇_U(Z))}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
(∇_V R(∇))(W,U)(Z) \to ∇_V(R(∇)(W,U)(Z)) - R(∇)(∇_V(W),U)(Z) - R(∇)(W,∇_V(U))(Z) - R(∇)(W,U)(∇_V(Z)) \quad \text{[(∇\_U R)(V,W)Z = ∇\_U R(V,W)Z \ensuremath{-} R(∇\_U V,W)Z \ensuremath{-} R(V,∇\_U W)Z \ensuremath{-} R(V,W) ∇\_U Z [∇]]\,(axiom)} \\
R(∇)(W,U)(Z) \to ∇_{W(∇_U(Z))} - ∇_{U(∇_W(Z))} - ∇_{[W,U]_{VF}(Z)} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
[W,U]_{VF} \to -[U,W]_{VF} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
∇_{(-[U,W]_{VF})(Z)} \to -∇_{[U,W]_{VF}(Z)} \quad \text{[∇\_X X-linearity [∇]]\,(axiom)} \\
∇_{V((∇_{W(∇_{U(Z)) + (-∇_{U(∇_{W(Z))) + (-(-∇_{[U,W]_{VF}(Z)))))}}}}}} \to ∇_{V(∇_{W(∇_U(Z)))}} - ∇_{V(∇_{U(∇_W(Z)))}} - ∇_{V((-∇_{[U,W]_{VF}(Z)))}} \quad \text{[∇\_X Y-additivity [∇]]\,(axiom)} \\
∇_{V((-∇_{[U,W]_{VF}(Z)))}} \to -∇_{V(∇_{[U,W]_{VF}(Z))}} \quad \text{[∇\_X Y-additivity [∇]]\,(axiom)} \\
R(∇)(∇_V(W),U)(Z) \to ∇_{∇_{V(W)(∇_U(Z))}} - ∇_{U(∇_{∇_V(W)(Z))}} - ∇_{[∇_{V(W),U]_{VF}(Z)}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
[∇_{V(W),U]_{VF}} \to -[U,∇_{V(W)]_{VF}} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
∇_{(-[U,∇_{V(W)]_{VF})(Z)}} \to -∇_{[U,∇_{V(W)]_{VF}(Z)}} \quad \text{[∇\_X X-linearity [∇]]\,(axiom)} \\
R(∇)(W,∇_V(U))(Z) \to ∇_{W(∇_{∇_V(U)(Z))}} - ∇_{∇_{V(U)(∇_W(Z))}} - ∇_{[W,∇_{V(U)]_{VF}(Z)}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
R(∇)(W,U)(∇_V(Z)) \to ∇_{W(∇_{U(∇_V(Z)))}} - ∇_{U(∇_{W(∇_V(Z)))}} - ∇_{[W,U]_{VF(∇_V(Z))}} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
[W,U]_{VF} \to -[U,W]_{VF} \quad \text{[[Y, X]\_VF \ensuremath{\to} \ensuremath{-}[X, Y]\_VF  [repr-canonical]]\,(axiom)} \\
∇_{(-[U,W]_{VF)(∇_V(Z))}} \to -∇_{[U,W]_{VF(∇_V(Z))}} \quad \text{[∇\_X X-linearity [∇]]\,(axiom)} \\
(∇_W R(∇))(U,V)(Z) \to ∇_W(R(∇)(U,V)(Z)) - R(∇)(∇_W(U),V)(Z) - R(∇)(U,∇_W(V))(Z) - R(∇)(U,V)(∇_W(Z)) \quad \text{[(∇\_U R)(V,W)Z = ∇\_U R(V,W)Z \ensuremath{-} R(∇\_U V,W)Z \ensuremath{-} R(V,∇\_U W)Z \ensuremath{-} R(V,W) ∇\_U Z [∇]]\,(axiom)} \\
R(∇)(U,V)(Z) \to ∇_{U(∇_V(Z))} - ∇_{V(∇_U(Z))} - ∇_{[U,V]_{VF}(Z)} \quad \text{[R(∇)(X,Y)Z = ∇\_X ∇\_Y Z \ensuremath{-} ∇\_Y ∇\_X Z \ensuremath{-} ∇\_[X,Y]\_VF Z [∇]]\,(axiom)} \\
∇_{W((

In [22]:
for i, step in enumerate(chain2.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] (∇_U R)(V,W)Z = ∇_U R(V,W)Z − R(∇_U V,W)Z − R(V,∇_U W)Z − R(V,W) ∇_U Z [∇] [axiom]
    (∇_U R(∇))(V,W)(Z)
 -> (∇_U(R(∇)(V,W)(Z)) + (-R(∇)(∇_U(V),W)(Z)) + (-R(∇)(V,∇_U(W))(Z)) + (-R(∇)(V,W)(∇_U(Z))))

[2] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(V,W)(Z)
 -> (∇_V(∇_W(Z)) + (-∇_W(∇_V(Z))) + (-∇_[V,W]_VF(Z)))

[3] ∇_X Y-additivity [∇] [axiom]
    ∇_U((∇_V(∇_W(Z)) + (-∇_W(∇_V(Z))) + (-∇_[V,W]_VF(Z))))
 -> (∇_U(∇_V(∇_W(Z))) + (-∇_U(∇_W(∇_V(Z)))) + (-∇_U(∇_[V,W]_VF(Z))))

[4] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(∇_U(V),W)(Z)
 -> (∇_∇_U(V)(∇_W(Z)) + (-∇_W(∇_∇_U(V)(Z))) + (-∇_[∇_U(V),W]_VF(Z)))

[5] [Y, X]_VF → −[X, Y]_VF  [repr-canonical] [axiom]
    [∇_U(V),W]_VF
 -> (-[W,∇_U(V)]_VF)

[6] ∇_X X-linearity [∇] [axiom]
    ∇_(-[W,∇_U(V)]_VF)(Z)
 -> (-∇_[W,∇_U(V)]_VF(Z))

[7] R(∇)(X,Y)Z = ∇_X ∇_Y Z − ∇_Y ∇_X Z − ∇_[X,Y]_VF Z [∇] [axiom]
    R(∇)(V,∇_U(W))(Z)
 -> (∇_V(∇_∇_U(W)(Z)) + (-∇_∇_U(W)(∇_V(Z))) + (-∇_[V,∇_U(W)]_VF(Z)))

[8

## Summary

- **Bianchi I**: $\sum_{cyc} R(U,V)W = \sum_{cyc}[(\nabla_U T)(V,W) + T(T(U,V),W)]$, closes syntactically to zero.
- **Bianchi II**: $\sum_{cyc} (\nabla_U R)(V,W)Z = \sum_{cyc} R(U, T(V,W)) Z$, closes syntactically to zero.

The heart of the closure: connection-linearity + tensor-Leibniz
definition reduce both sides entirely to $\nabla$ and LBVF terms; then
LBVF arg-Sum/Neg linearity together with arg-antisymmetry pull terms
into canonical order; finally the *LBVF Jacobi cyclic-triple* rule
erases the three-term residue at the `Sum` level. `_extract_bracket_with_wrapper`
was extended for `ConnectionEvalExpr` so $\nabla_{[A,[B,C]_{VF}]_{VF}}(Z) +$
cycle is recognised as a Jacobi shape.